In [1]:
# Food Additive Embeddings Generator for Google Colab (Hand-crafted Embeddings)
# Run this in Google Colab to generate embeddings for your CSV data

import pandas as pd
import numpy as np
import json
import math
from google.colab import files

# Upload your CSV files
print("Please upload your food additive CSV files...")
uploaded = files.upload()

EMBEDDING_DIMENSION = 384

def generate_query_embedding(text):
    """
    Generate hand-crafted embedding based on text characteristics
    This matches the Swift implementation in your iOS app
    """
    # Create embedding array filled with zeros
    embedding = [0.0] * EMBEDDING_DIMENSION

    # Clean and split text
    clean_text = text.lower()
    words = [word.strip() for word in clean_text.split() if word.strip()]

    if not words:
        return embedding

    # Generate features based on text characteristics
    for word_index, word in enumerate(words):
        word_weight = 1.0 / len(words)
        position_weight = 1.0 - (word_index / len(words))

        # Character-based features
        for char_index, char in enumerate(word):
            ascii_val = ord(char)
            embedding_index = (char_index + word_index * 10) % EMBEDDING_DIMENSION
            embedding[embedding_index] += math.sin(ascii_val * 0.1) * word_weight * position_weight

        # Word length features
        length = len(word)
        length_index = (length * 7) % EMBEDDING_DIMENSION
        embedding[length_index] += math.tanh(length / 10.0) * word_weight

        # Chemical pattern features
        chemical_patterns = ["acid", "ate", "ine", "ium", "ide", "oxy", "meth", "eth", "prop"]
        for pattern_index, pattern in enumerate(chemical_patterns):
            if pattern in word:
                pattern_embedding_index = (pattern_index * 23 + word_index * 5) % EMBEDDING_DIMENSION
                embedding[pattern_embedding_index] += 0.5 * word_weight

    # Normalize the embedding
    magnitude = math.sqrt(sum(x * x for x in embedding))
    if magnitude > 0:
        embedding = [x / magnitude for x in embedding]

    return embedding

# Process each uploaded CSV file
additive_data = []

for filename in uploaded.keys():
    print(f"\nProcessing {filename}...")
    df = pd.read_csv(filename)

    # Check if this is the substances CSV (has Substance, Other Names, Used for columns)
    if 'Substance' in df.columns and 'Other Names' in df.columns:
        print("Processing substances CSV...")
        for _, row in df.iterrows():
            substance = str(row['Substance']).strip()
            other_names = str(row['Other Names']).strip() if pd.notna(row['Other Names']) else ""
            technical_effect = str(row['Used for (Technical Effect)']).strip() if pd.notna(row['Used for (Technical Effect)']) else ""

            if substance and substance != 'nan':
                # Create searchable text combining all information
                searchable_text = f"{substance} {other_names} {technical_effect}".strip()

                additive_data.append({
                    'substance': substance,
                    'other_names': other_names,
                    'technical_effect': technical_effect,
                    'searchable_text': searchable_text
                })

print(f"\nTotal additive records: {len(additive_data)}")

# Remove duplicates based on substance name
seen_substances = set()
unique_data = []
for item in additive_data:
    substance_lower = item['substance'].lower()
    if substance_lower not in seen_substances:
        seen_substances.add(substance_lower)
        unique_data.append(item)

print(f"Unique additive records: {len(unique_data)}")

# Generate embeddings using hand-crafted method
print("\nGenerating hand-crafted embeddings...")
embeddings_list = []

for i, item in enumerate(unique_data):
    if i % 100 == 0:
        print(f"Processed {i}/{len(unique_data)} records...")

    embedding = generate_query_embedding(item['searchable_text'])
    embeddings_list.append(embedding)

print(f"Generated {len(embeddings_list)} embeddings")

# Create the final data structure
final_data = []
for i, item in enumerate(unique_data):
    final_data.append({
        'substance': item['substance'],
        'other_names': item['other_names'],
        'technical_effect': item['technical_effect'],
        'searchable_text': item['searchable_text'],
        'embedding': embeddings_list[i]
    })

# Save as JSON
output_filename = 'food_additive_embeddings.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'hand-crafted',
        'embedding_dimension': EMBEDDING_DIMENSION,
        'total_records': len(final_data),
        'data': final_data
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved embeddings to {output_filename}")
print(f"Embedding dimension: {EMBEDDING_DIMENSION}")
print(f"File size: {len(json.dumps(final_data)) / 1024 / 1024:.2f} MB")

# Download the file
files.download(output_filename)

Please upload your food additive CSV files...


Saving FoodSubstances_cleaned_v2.csv to FoodSubstances_cleaned_v2.csv

Processing FoodSubstances_cleaned_v2.csv...
Processing substances CSV...

Total additive records: 3971
Unique additive records: 3971

Generating hand-crafted embeddings...
Processed 0/3971 records...
Processed 100/3971 records...
Processed 200/3971 records...
Processed 300/3971 records...
Processed 400/3971 records...
Processed 500/3971 records...
Processed 600/3971 records...
Processed 700/3971 records...
Processed 800/3971 records...
Processed 900/3971 records...
Processed 1000/3971 records...
Processed 1100/3971 records...
Processed 1200/3971 records...
Processed 1300/3971 records...
Processed 1400/3971 records...
Processed 1500/3971 records...
Processed 1600/3971 records...
Processed 1700/3971 records...
Processed 1800/3971 records...
Processed 1900/3971 records...
Processed 2000/3971 records...
Processed 2100/3971 records...
Processed 2200/3971 records...
Processed 2300/3971 records...
Processed 2400/3971 recor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>